# Big Five — retrain on local (GPU) embeddings

Run this on a **cloud GPU** (Colab: Runtime -> Change runtime type -> GPU).

It will:
1. Install dependencies
2. Download the raw `essays-big5` corpus (text + O/C/E/A/N labels)
3. Embed every essay with a local `sentence-transformers` model on GPU
4. Train the regularized linear classifier (5 logistic regressions)
5. Evaluate honestly on the held-out test split vs. the Mistral baseline (0.6202)
6. Save `personality_model.pt` + `x_scaler.pkl` and zip them for download

Drop the downloaded artifacts into the repo's `artifacts/` folder and the FastAPI service uses them as-is.

## 1. Install dependencies

In [1]:
!pip -q install "sentence-transformers>=2.7" "datasets>=2.16" "scikit-learn>=1.3" "torch>=2.0" joblib numpy pandas

In [2]:
import torch
print('torch', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU (embedding will be slow)')

torch 2.10.0+cu128
CUDA available: True
device: Tesla T4


## 2. Config

`EMBED_MODEL` is the local embedder. `all-mpnet-base-v2` (768-dim) is a strong general default.
You can swap in a bigger model (e.g. `intfloat/e5-large-v2`, 1024-dim) to try for more accuracy.

In [3]:
EMBED_MODEL = 'intfloat/e5-large-v2'   # 1024-dim; stronger than mpnet (mpnet gave 0.563)
# Alternatives to try in this same slot:
#   'all-mpnet-base-v2'      # 768-dim, gave 0.563
#   'BAAI/bge-large-en-v1.5' # 1024-dim, also strong (same 'query:' prefix style as e5)
DATASET     = 'jingjietan/essays-big5'   # raw text + labels
TRAIT_KEYS  = ['O', 'C', 'E', 'A', 'N']
TRAIT_NAMES = ['Openness', 'Conscientiousness', 'Extraversion', 'Agreeableness', 'Neuroticism']
C_GRID      = [0.003, 0.01, 0.03, 0.1, 0.3, 1.0]
NORMALIZE   = True   # L2-normalize embeddings (recommended for cosine-style models)
MISTRAL_BASELINE = 0.6202

# e5 / bge models are trained WITH an instruction prefix and score badly without it.
# We prepend 'query: ' for those models only; others get no prefix.
EMBED_PREFIX = 'query: ' if any(k in EMBED_MODEL.lower() for k in ('e5', 'bge')) else ''
print('EMBED_MODEL =', EMBED_MODEL, '| prefix =', repr(EMBED_PREFIX))

EMBED_MODEL = intfloat/e5-large-v2 | prefix = 'query: '


## 3. Load raw essays (text + labels)

In [4]:
import numpy as np
from datasets import load_dataset

ds = load_dataset(DATASET)
print('splits:', list(ds.keys()))

def texts_labels(split):
    texts = [str(t) for t in ds[split]['text']]
    y = np.column_stack([np.array(ds[split][k], dtype=np.float32) for k in TRAIT_KEYS])
    return texts, y

tr_t, ytr = texts_labels('train')
va_t, yva = texts_labels('validation')
te_t, yte = texts_labels('test')
print(f'train {len(tr_t)} | val {len(va_t)} | test {len(te_t)}')
print('example:', tr_t[0][:150])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/3.13M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/795k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/953k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1578 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/395 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/494 [00:00<?, ? examples/s]

splits: ['train', 'validation', 'test']
train 1578 | val 395 | test 494
example: it is wednesday. I can't wait until friday because I am going home to see brandon. I miss him so much. I can't wait to see him. two more days. this ha


## 4. Embed on GPU

In [5]:
from sentence_transformers import SentenceTransformer

device = 'cuda' if torch.cuda.is_available() else 'cpu'
encoder = SentenceTransformer(EMBED_MODEL, device=device)
EMBED_DIM = encoder.get_sentence_embedding_dimension()
print(f'{EMBED_MODEL} -> {EMBED_DIM}-dim, device={device}')

def embed(texts):
    prefixed = [EMBED_PREFIX + t for t in texts]
    return encoder.encode(prefixed, batch_size=64, show_progress_bar=True,
                          normalize_embeddings=NORMALIZE, convert_to_numpy=True)

Xtr = embed(tr_t); Xva = embed(va_t); Xte = embed(te_t)
print('shapes:', Xtr.shape, Xva.shape, Xte.shape)

modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/616 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.34G [00:00<?, ?B/s]

...

/tmp/ipykernel_58/3602596944.py:5: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  EMBED_DIM = encoder.get_sentence_embedding_dimension()


Batches:   0%|          | 0/25 [00:00<?, ?it/s]

Batches:   0%|          | 0/7 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

shapes: (1578, 1024) (395, 1024) (494, 1024)


## 5. Train the linear classifier

Same design as the production model: scale inputs, fit one L2-penalized logistic
regression per trait (C picked on validation), refit on train+val, fold into a
single `nn.Linear` so the saved checkpoint matches the FastAPI service's `PersonalityClassifier`.

In [6]:
import torch.nn as nn
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

class PersonalityClassifier(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc_out = nn.Linear(input_dim, 5)
    def forward(self, x):
        return self.fc_out(x)

Xfit = np.vstack([Xtr, Xva]); yfit = np.vstack([ytr, yva])
scaler = StandardScaler().fit(Xfit)
Xf, Xt = scaler.transform(Xfit), scaler.transform(Xte)
Xtr_s, Xva_s = scaler.transform(Xtr), scaler.transform(Xva)

W = np.zeros((5, EMBED_DIM), dtype=np.float32); b = np.zeros(5, dtype=np.float32); chosen_C = []
for i, name in enumerate(TRAIT_NAMES):
    best_acc, best_C = -1.0, None
    for C in C_GRID:
        clf = LogisticRegression(C=C, max_iter=3000).fit(Xtr_s, ytr[:, i])
        v = accuracy_score(yva[:, i], clf.predict(Xva_s))
        if v > best_acc: best_acc, best_C = v, C
    clf = LogisticRegression(C=best_C, max_iter=3000).fit(Xf, yfit[:, i])
    W[i] = clf.coef_[0]; b[i] = clf.intercept_[0]; chosen_C.append(best_C)
    print(f'  {name:18s} C={best_C:<6} val_acc={best_acc:.4f}')

model = PersonalityClassifier(EMBED_DIM)
with torch.no_grad():
    model.fc_out.weight.copy_(torch.tensor(W))
    model.fc_out.bias.copy_(torch.tensor(b))
model.eval()

  Openness           C=0.003  val_acc=0.5924
  Conscientiousness  C=0.003  val_acc=0.5772
  Extraversion       C=0.003  val_acc=0.5873
  Agreeableness      C=0.003  val_acc=0.5772
  Neuroticism        C=0.003  val_acc=0.5747


PersonalityClassifier(
  (fc_out): Linear(in_features=1024, out_features=5, bias=True)
)

## 6. Honest evaluation on the held-out test split

In [8]:
with torch.no_grad():
    probs = torch.sigmoid(model(torch.tensor(Xt, dtype=torch.float32))).numpy()
preds = (probs >= 0.5).astype(int)

trait_accs = [accuracy_score(yte[:, i], preds[:, i]) for i in range(5)]
trait_aucs = [roc_auc_score(yte[:, i], probs[:, i]) for i in range(5)]
test_acc = accuracy_score(yte.flatten(), preds.flatten())

print(f'{"Trait":18s} {"acc":>6} {"AUC":>6}')
for n, a, u in zip(TRAIT_NAMES, trait_accs, trait_aucs):
    print(f'{n:18s} {a:6.3f} {u:6.3f}')
print(f'\nMEAN test acc : {np.mean(trait_accs):.4f}')
print(f'Mistral baseline: {MISTRAL_BASELINE:.4f}')
delta = np.mean(trait_accs) - MISTRAL_BASELINE
print(f'Delta          : {delta:+.4f}  ->', 'LOCAL WINS/TIES, ship it' if delta >= -0.005 else 'local is worse; see note below')

Trait                 acc    AUC
Openness            0.630  0.655
Conscientiousness   0.567  0.577
Extraversion        0.571  0.577
Agreeableness       0.587  0.597
Neuroticism         0.589  0.609

MEAN test acc : 0.5887
Mistral baseline: 0.6202
Delta          : -0.0315  -> local is worse; see note below


## 7. Save artifacts (compatible with the FastAPI service)

In [9]:
import joblib, os
os.makedirs('artifacts', exist_ok=True)

joblib.dump(scaler, 'artifacts/x_scaler.pkl')
torch.save({
    'model_state_dict': model.state_dict(),
    'input_dim': EMBED_DIM,
    'test_accuracy': test_acc,
    'trait_names': TRAIT_NAMES,
    'trait_accuracies': trait_accs,
    'model_type': f'logistic_regression_linear__{EMBED_MODEL}',
    'embed_model': EMBED_MODEL,
    'embed_dim': EMBED_DIM,
    'embed_prefix': EMBED_PREFIX,
    'normalize_embeddings': NORMALIZE,
    'chosen_C': chosen_C,
}, 'artifacts/personality_model.pt')
print('saved artifacts/personality_model.pt + artifacts/x_scaler.pkl')

saved artifacts/personality_model.pt + artifacts/x_scaler.pkl


In [10]:
# Zip for download
import shutil
shutil.make_archive('personality_artifacts', 'zip', 'artifacts')
print('created personality_artifacts.zip')
try:
    from google.colab import files
    files.download('personality_artifacts.zip')
except Exception:
    print('Not on Colab — download personality_artifacts.zip from the file browser.')

created personality_artifacts.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 8. Quick sanity check — predict from raw text

This mirrors exactly what the FastAPI service does at inference time.

In [11]:
INTERP = {
    'Openness': {0: 'Practical, conventional', 1: 'Creative, open to new experiences'},
    'Conscientiousness': {0: 'Spontaneous, flexible', 1: 'Organized, disciplined'},
    'Extraversion': {0: 'Reserved, introverted', 1: 'Outgoing, energetic'},
    'Agreeableness': {0: 'Competitive, skeptical', 1: 'Cooperative, compassionate'},
    'Neuroticism': {0: 'Emotionally stable', 1: 'Sensitive, anxious'},
}

def predict_text(text):
    emb = encoder.encode([EMBED_PREFIX + text], normalize_embeddings=NORMALIZE, convert_to_numpy=True)
    scaled = scaler.transform(emb)
    with torch.no_grad():
        p = torch.sigmoid(model(torch.tensor(scaled, dtype=torch.float32)))[0].numpy()
    for i, t in enumerate(TRAIT_NAMES):
        hi = int(p[i] >= 0.5)
        print(f'{t:18s} {"High" if hi else "Low "} ({p[i]:.0%})  {INTERP[t][hi]}')

predict_text('I love meeting new people, throwing parties, and trying wild new experiences.')
print()
predict_text('I prefer quiet evenings alone with a book and a strict daily routine.')

Openness           High (93%)  Creative, open to new experiences
Conscientiousness  Low  (19%)  Spontaneous, flexible
Extraversion       High (95%)  Outgoing, energetic
Agreeableness      High (64%)  Cooperative, compassionate
Neuroticism        High (56%)  Sensitive, anxious

Openness           High (87%)  Creative, open to new experiences
Conscientiousness  High (60%)  Organized, disciplined
Extraversion       Low  (21%)  Reserved, introverted
Agreeableness      High (63%)  Cooperative, compassionate
Neuroticism        Low  (45%)  Emotionally stable


## Next steps (back in the repo)

1. Unzip `personality_artifacts.zip` into the repo's `artifacts/` folder (overwrite the Mistral ones).
2. The service must embed with the **same** model. Tell Claude the `EMBED_MODEL` and `EMBED_DIM` you used so it wires the `LocalEmbedder` in as the default and updates config/tests.
3. If local accuracy was **worse** than the Mistral baseline: try a larger embedder (e.g. `intfloat/e5-large-v2`) in cell 2, or keep the Mistral model and use the `/predict/embedding` endpoint. Bigger nets won't help — the ceiling is the input, not the classifier.